# Advanced Problems with Solutions: `decimal.Decimal` Contexts, Precision, and Rounding

These problems focus on precise decimal arithmetic, global vs local contexts, rounding modes, `quantize`, traps, flags, and best practices.

In [1]:
import decimal
from decimal import Decimal

## Problem 1: Avoiding Binary Float Contamination

You are processing financial values. Explain and demonstrate the difference between these two constructions:

```python
Decimal(0.1)
Decimal('0.1')
```

Then compute `0.1 + 0.2` correctly using `Decimal`.

In [2]:
# Solution

a = Decimal(0.1)
b = Decimal('0.1')

print(a)
print(b)

result = Decimal('0.1') + Decimal('0.2')
print(result)

0.1000000000000000055511151231257827021181583404541015625
0.1
0.3


`Decimal(0.1)` receives an already-imprecise binary floating-point value. `Decimal('0.1')` receives an exact decimal string.

Best practice: construct `Decimal` objects from strings or integers, not floats.

## Problem 2: Local Context Isolation

Set the global precision to `28`. Inside a local context, set precision to `5` and compute `1 / 7`. After leaving the local context, prove that the global precision is unchanged.

In [3]:
# Solution

decimal.getcontext().prec = 28

print('Global precision before:', decimal.getcontext().prec)

with decimal.localcontext() as ctx:
    ctx.prec = 5
    value = Decimal('1') / Decimal('7')
    print('Local result:', value)
    print('Local precision:', ctx.prec)

print('Global precision after:', decimal.getcontext().prec)

Global precision before: 28
Local result: 0.14286
Local precision: 5
Global precision after: 28


A local context is a temporary copy of the active context. Mutating it does not mutate the global context.

## Problem 3: Rounding Half Even vs Half Up

Round the following values to one decimal place using both `ROUND_HALF_EVEN` and `ROUND_HALF_UP`:

```python
1.25, 1.35, 1.45, 1.55
```

Explain why the answers differ.

In [4]:
# Solution

values = [Decimal('1.25'), Decimal('1.35'), Decimal('1.45'), Decimal('1.55')]

with decimal.localcontext() as ctx:
    ctx.rounding = decimal.ROUND_HALF_EVEN
    print('ROUND_HALF_EVEN')
    for x in values:
        print(x, '->', round(x, 1))

print()

with decimal.localcontext() as ctx:
    ctx.rounding = decimal.ROUND_HALF_UP
    print('ROUND_HALF_UP')
    for x in values:
        print(x, '->', round(x, 1))

ROUND_HALF_EVEN
1.25 -> 1.2
1.35 -> 1.4
1.45 -> 1.4
1.55 -> 1.6

ROUND_HALF_UP
1.25 -> 1.3
1.35 -> 1.4
1.45 -> 1.5
1.55 -> 1.6


`ROUND_HALF_EVEN` rounds ties toward the nearest even final digit. This reduces cumulative rounding bias.

`ROUND_HALF_UP` rounds ties away from zero in the usual schoolbook style.

## Problem 4: Currency Quantization

A store computes tax as `price * tax_rate`. Given:

```python
price = Decimal('19.995')
tax_rate = Decimal('0.0825')
```

Compute the final price rounded to exactly two decimal places using `ROUND_HALF_UP`.

In [5]:
# Solution

price = Decimal('19.995')
tax_rate = Decimal('0.0825')
cent = Decimal('0.01')

with decimal.localcontext() as ctx:
    ctx.rounding = decimal.ROUND_HALF_UP
    tax = (price * tax_rate).quantize(cent)
    final_price = (price + tax).quantize(cent)

print('Tax:', tax)
print('Final price:', final_price)

Tax: 1.65
Final price: 21.65


`quantize(Decimal('0.01'))` is the preferred way to force a fixed number of decimal places for money-like values.

## Problem 5: Precision Does Not Mean Decimal Places

Set precision to `4` and compute:

```python
Decimal('12345') + Decimal('1')
Decimal('1.2345') + Decimal('0.0001')
```

Explain the results.

In [6]:
# Solution

with decimal.localcontext() as ctx:
    ctx.prec = 4
    print(Decimal('12345') + Decimal('1'))
    print(Decimal('1.2345') + Decimal('0.0001'))

1.235E+4
1.235


Context precision controls significant digits, not the number of digits after the decimal point.

## Problem 6: Detecting Inexact Results with Flags

Use a local context with precision `10` to compute `1 / 8` and `1 / 7`. Inspect whether each operation was exact using context flags.

In [7]:
# Solution

with decimal.localcontext() as ctx:
    ctx.prec = 10
    ctx.clear_flags()
    exact_value = Decimal('1') / Decimal('8')
    print('1 / 8 =', exact_value)
    print('Inexact?', ctx.flags[decimal.Inexact])

    ctx.clear_flags()
    inexact_value = Decimal('1') / Decimal('7')
    print('1 / 7 =', inexact_value)
    print('Inexact?', ctx.flags[decimal.Inexact])

1 / 8 = 0.125
Inexact? False
1 / 7 = 0.1428571429
Inexact? True


`1 / 8` terminates exactly in decimal form. `1 / 7` does not, so the `Inexact` flag is set.

## Problem 7: Turning Inexact Results into Exceptions

Modify the previous problem so that an inexact result raises an exception.

In [8]:
# Solution

with decimal.localcontext() as ctx:
    ctx.prec = 10
    ctx.traps[decimal.Inexact] = True

    print(Decimal('1') / Decimal('8'))

    try:
        print(Decimal('1') / Decimal('7'))
    except decimal.Inexact:
        print('Inexact result detected and blocked.')

0.125
Inexact result detected and blocked.


Traps are useful when silent rounding would be dangerous, such as in auditing or validation logic.

## Problem 8: Safe Division Function

Write a function `safe_divide(a, b, precision)` that:

- accepts string inputs,
- uses a local decimal context,
- sets the requested precision,
- raises `ZeroDivisionError` for division by zero,
- returns a `Decimal` result.

In [9]:
# Solution

def safe_divide(a, b, precision):
    x = Decimal(a)
    y = Decimal(b)

    if y == 0:
        raise ZeroDivisionError('Cannot divide by zero.')

    with decimal.localcontext() as ctx:
        ctx.prec = precision
        return x / y


print(safe_divide('1', '7', 6))
print(safe_divide('10.5', '0.25', 10))

0.142857
42


The function avoids global context mutation and keeps decimal creation exact by using strings.

## Problem 9: Compound Interest with Decimal Precision

Compute the future value of an investment using:

```python
principal = Decimal('1000.00')
annual_rate = Decimal('0.0575')
years = 10
compounds_per_year = 12
```

Round the final answer to cents using `ROUND_HALF_EVEN`.

In [10]:
# Solution

principal = Decimal('1000.00')
annual_rate = Decimal('0.0575')
years = 10
compounds_per_year = 12

cent = Decimal('0.01')

with decimal.localcontext() as ctx:
    ctx.prec = 40
    ctx.rounding = decimal.ROUND_HALF_EVEN

    n = Decimal(compounds_per_year)
    periods = compounds_per_year * years
    future_value = principal * (Decimal('1') + annual_rate / n) ** periods
    future_value = future_value.quantize(cent)

print(future_value)

1774.69


Intermediate precision is intentionally higher than the final display precision. This reduces premature rounding error.

## Problem 10: Auditable Invoice Calculation

Given line items with quantity, unit price, and discount rate, compute an invoice total. Each line subtotal should be rounded to cents using `ROUND_HALF_UP`. The invoice total should be the sum of rounded line subtotals.

```python
items = [
    {'qty': '3', 'unit_price': '19.995', 'discount': '0.10'},
    {'qty': '2', 'unit_price': '5.555', 'discount': '0.00'},
    {'qty': '7', 'unit_price': '1.005', 'discount': '0.05'}
]
```

In [11]:
# Solution

items = [
    {'qty': '3', 'unit_price': '19.995', 'discount': '0.10'},
    {'qty': '2', 'unit_price': '5.555', 'discount': '0.00'},
    {'qty': '7', 'unit_price': '1.005', 'discount': '0.05'}
]

cent = Decimal('0.01')

with decimal.localcontext() as ctx:
    ctx.prec = 28
    ctx.rounding = decimal.ROUND_HALF_UP

    rounded_subtotals = []

    for item in items:
        qty = Decimal(item['qty'])
        unit_price = Decimal(item['unit_price'])
        discount = Decimal(item['discount'])

        raw_subtotal = qty * unit_price * (Decimal('1') - discount)
        rounded_subtotal = raw_subtotal.quantize(cent)
        rounded_subtotals.append(rounded_subtotal)

        print('Raw:', raw_subtotal, 'Rounded:', rounded_subtotal)

    invoice_total = sum(rounded_subtotals, Decimal('0.00'))

print('Invoice total:', invoice_total)

Raw: 53.98650 Rounded: 53.99
Raw: 11.11000 Rounded: 11.11
Raw: 6.68325 Rounded: 6.68
Invoice total: 71.78


This approach is auditable because every rounded line subtotal is visible and the final total is the sum of those rounded values.

## Best Practices Summary

- Prefer `Decimal('...')` over `Decimal(float)`.
- Use `localcontext()` instead of mutating the global context.
- Use `quantize()` for fixed decimal places.
- Use higher internal precision than final display precision.
- Use flags and traps when rounding or inexact arithmetic must be detected.
- Be explicit about rounding rules in financial or regulatory code.